# Retrieved Context Token Counter (CSV)
Counts tokens in the `retrieved_contexts` column across all CSV files in a folder, concurrently.

No need to install GPT-4o itself — this only pulls the tokenizer (via `litellm`), which happens automatically.

In [ ]:
# If needed (usually already installed):
# !pip install litellm pandas --quiet

In [ ]:
import ast
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
from litellm import token_counter

## Config — just set your folder path here

In [ ]:
FOLDER_PATH = "./my_csv_files"     # <-- point this at your folder of .csv files
COLUMN = "retrieved_contexts"      # column to count tokens for
MODEL = "gpt-4o"                   # tokenizer to use
WORKERS = 4                        # how many files to process in parallel
OUTPUT_DIR = "./token_outputs"     # where results get saved

## Helper functions

In [ ]:
def parse_contexts(raw):
    """retrieved_contexts is stored as a stringified list, e.g. \"['a', 'b']\".
    Falls back to a single plain string if parsing fails."""
    if pd.isna(raw):
        return []
    if isinstance(raw, list):
        return raw
    try:
        val = ast.literal_eval(str(raw))
        return val if isinstance(val, list) else [str(val)]
    except (ValueError, SyntaxError):
        return [str(raw)]


def count_row_tokens(contexts, model):
    if not contexts:
        return 0, 0
    per_chunk = [token_counter(model=model, text=c) for c in contexts]
    return sum(per_chunk), len(contexts)


def process_file(path: Path, column: str, model: str, output_dir: Path):
    df = pd.read_csv(path)
    if column not in df.columns:
        raise ValueError(f"Column '{column}' not found in {path.name}. Found: {list(df.columns)}")

    token_counts, chunk_counts = [], []
    for raw in df[column]:
        contexts = parse_contexts(raw)
        total_tokens, n_chunks = count_row_tokens(contexts, model)
        token_counts.append(total_tokens)
        chunk_counts.append(n_chunks)

    df[f"{column}_token_count"] = token_counts
    df[f"{column}_num_chunks"] = chunk_counts

    out_path = output_dir / f"{path.stem}_tokens{path.suffix}"
    df.to_csv(out_path, index=False)

    return {
        "file": path.name,
        "output": out_path.name,
        "rows": len(df),
        "total_tokens": sum(token_counts),
        "avg_tokens_per_row": round(sum(token_counts) / len(df), 1) if len(df) else 0,
        "max_tokens_row": max(token_counts) if token_counts else 0,
    }

## Run — picks up every `.csv` file in `FOLDER_PATH`

In [ ]:
paths = sorted(Path(FOLDER_PATH).glob("*.csv"))
print(f"Found {len(paths)} file(s):")
for p in paths:
    print(" -", p.name)

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
results = []
with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    futures = {
        pool.submit(process_file, p, COLUMN, MODEL, output_dir): p
        for p in paths
    }
    for fut in as_completed(futures):
        path = futures[fut]
        try:
            results.append(fut.result())
        except Exception as e:
            print(f"FAILED: {path.name} -> {e}")

## Summary

In [ ]:
summary_df = pd.DataFrame(results).sort_values("file").reset_index(drop=True)
summary_df